# Key Node Phylostratigraphy

In [1]:
from collections import Counter, defaultdict
from dendropy import Tree
from tqdm.auto import tqdm
import itertools
import os
import pandas as pd
import pyham
import sys

In [2]:
fastoma_results_path = './result'

In [3]:
# load the tree and get all the branches
nwk_fn = os.path.join(fastoma_results_path, 'species_tree_checked_relabelled.nwk')
t = Tree.get(path=nwk_fn, schema='newick')

key_nodes = list(filter(lambda n: '.' not in n.label, t.internal_nodes()))
species_level_paths = defaultdict(list)
onestep_higher_paths = defaultdict(list)

for n in key_nodes:
    # get species below
    for leaf in n.leaf_nodes():
        path_len = (leaf.distance_from_root() - n.distance_from_root())
        species_level_paths[n.label].append((n.label, leaf.taxon.label, path_len))

    # find species and then get set of nodes one step higher
    for n2 in set(map(lambda n1: n1.parent_node, n.leaf_nodes())):
        if n.label != n2.label:
            path_len = (n2.distance_from_root() - n.distance_from_root())
            onestep_higher_paths[n.label].append((n.label, n2.label, path_len))

# normalise the differences in branch length between the paths
for k, v in species_level_paths.items():
    longest = max(map(lambda x: x[2], v))
    for i in range(len(v)):
        species_level_paths[k][i] = (*species_level_paths[k][i], species_level_paths[k][i][2] / longest)

for k, v in onestep_higher_paths.items():
    longest = max(map(lambda x: x[2], v))
    for i in range(len(v)):
        onestep_higher_paths[k][i] = (*onestep_higher_paths[k][i], onestep_higher_paths[k][i][2] / longest)

In [4]:
# orthoxml
ham = pyham.Ham(
            tree_file=nwk_fn,
            tree_format="newick",
            hog_file=os.path.join(fastoma_results_path, 'FastOMA_HOGs.orthoxml'),
            type_hog_file="orthoxml",
            filter_object=None,
            use_internal_name=True,
            with_parser_progress=True,
        )

Parsing Species: 0it [00:00, ?it/s]

Parsing HOGs: 0it [00:00, ?it/s]

NOTE: we only want to include SOME HOGs in our final method.

In [5]:
with open('./hogs_to_keep_no_misplaced.txt', 'rt') as fp:
    hogs_to_use = set(map(lambda x: x.rstrip(), fp.readlines()))

In [7]:
def filter_hogs(g, hogs_to_use):
    return ((not g.is_singleton()) and (g.get_top_level_hog().hog_id in hogs_to_use))

def get_count(gs):
    return sum(1 for _ in filter(lambda g: filter_hogs(g, hogs_to_use), gs))

def do(path_dict):
    for key_node_label in tqdm(path_dict):
        paths = path_dict[key_node_label]
        for i in tqdm(range(len(paths))):
            tail_genome = ham.get_ancestral_genome_by_name(paths[i][0])
            try:
                head_genome = ham.get_ancestral_genome_by_name(paths[i][1])
            except:
                head_genome = ham.get_extant_genome_by_name(paths[i][1])
            vmap = ham.compare_genomes_vertically(head_genome, tail_genome)
    
            res = {'key_node_label': key_node_label,
                   'parent_genome_name': tail_genome.name,
                   'child_genome_name': head_genome.name,
                   'retained': get_count(vmap.get_retained().keys()),
                   'gained': get_count(vmap.get_gained()),
                   'duplicated_child_count': get_count(itertools.chain.from_iterable(vmap.get_duplicated().values())),
                   'duplicated_parent_count': get_count(vmap.get_duplicated().keys()),
                   'lost': get_count(vmap.get_lost()),
                   'parent_genome_size': get_count(tail_genome.genes),
                   'child_genome_size': get_count(head_genome.genes),
                   'path_len': paths[i][2],
                   'normalised_path_len': paths[i][3]}
            yield res

In [8]:
species_level_paths_df = pd.DataFrame(do(species_level_paths))

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

In [9]:
onestep_higher_paths_df = pd.DataFrame(do(onestep_higher_paths))

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
tree_name = '_'.join(os.path.basename(os.getcwd()).split('_')[2:])
class_df = pd.read_csv(f'../labelled_trees/{tree_name}_colouring.tsv', sep='\t')

In [11]:
def add_child_classification(df):
    df1 = pd.merge(df, class_df, left_on='child_genome_name', right_on='id')
    header = list(df.keys())[1:]
    header = ['key_node_label', 'classification'] + header
    return df1[header].rename(columns={'classification': 'child_classification'})

In [12]:
species_level_paths_df = add_child_classification(species_level_paths_df)
onestep_higher_paths_df = add_child_classification(onestep_higher_paths_df)

In [13]:
species_level_paths_df.to_csv('results_loss_analysis/species_level_paths.tsv', sep='\t', index=False)
onestep_higher_paths_df.to_csv('results_loss_analysis/onestep_higher_paths.tsv', sep='\t', index=False)